```python
# Copyright (c) 2026 Louis Gevers
# SPDX-License-Identifier: BSD-3-Clause
```

# Legged Robots Project - Task 3: DRL on Stairs

**Goal:** train a policy that makes the Go2 walk forward up and down a stairs reliably.

A flat-terrain policy fails on stairs as it has neve seen a step, and cannot see one coming.
This notebook changes the terrain to stairs, and introduces new components to train a more robust policy:

| Component | Why | Section |
| --- | --- | --- |
| Terrain scan | the policy perceives the steps ahead (exteroception) | 3.1b |
| Observation noise | real sensors are noisy, a policy trained on perfect data may not transfer | 3.1b |
| Domain randomization | the real robot never exactly matches the simulation model | 3.1f |
| Curriculum learning | robots start on easy stairs and only get harder ones once they succeed | 3.1g |


Start from your Task 1 configuration and adapt it: most of your observations, actions, rewards and terminations carry over!
There is no analysis notebook for this task: the goal is a policy that crosses stairs reliably, checked qualitatively in the viewer (3.4).

**Time budget.** Stairs are harder than flat ground and training takes longer!
Check your environment in the viewer (3.1i) before training, and plan your GPU time, especially close to the deadline.

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import copy
import lr_project # Before mjlab import

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.rl import RslRlModelCfg, RslRlPpoAlgorithmCfg, RslRlOnPolicyRunnerCfg

from mjlab.managers.action_manager import ActionTermCfg
from mjlab.managers.command_manager import CommandTermCfg
from mjlab.managers.curriculum_manager import CurriculumTermCfg
from mjlab.managers.event_manager import EventTermCfg
from mjlab.managers.observation_manager import ObservationTermCfg, ObservationGroupCfg
from mjlab.managers.reward_manager import RewardTermCfg
from mjlab.managers.termination_manager import TerminationTermCfg
from mjlab.managers.scene_entity_config import SceneEntityCfg
from mjlab.utils.noise import UniformNoiseCfg

from lr_project import mdp, scene, play, train, evaluate, registry

### Run configuration

Two runs are planned. Run 1 asks whether the robot can climb at all: flat MDP plus the terrain scan and relaxed rewards, without noise or domain randomization. Run 2 is the final policy: noise on the actor observations and randomized base centre of mass. Both are coded below and switched on only through these flags, so the two runs differ by exactly these settings.

In [ ]:
import yaml

USE_NOISE = False  # uniform noise on the actor observations (3.1b)
USE_DR = False  # randomized base centre of mass (3.1f)

# Log folder name, built from the flags: "scan", "scan_noise_dr", ...
VARIANT = "scan" + ("_noise" if USE_NOISE else "") + ("_dr" if USE_DR else "")

# One seed per configuration: a stairs run takes about 15 min and this notebook has
# no controlled experiment, so the GPU time goes to the configurations themselves
SEEDS = [42]
print(f"variant: {VARIANT}, seeds: {SEEDS}")

## 3.1 Building the environment configuration

The structure is the same as in Task 1.
The scene now contains a generated stairs terrain and a terrain scan sensor (see `src/lr_project/scene.py`), and the environment gets one new part: the curriculum (3.1g).

### 3.1a Commands

This task is **forward locomotion only**: `rel_forward_envs=1.0` gives every robot a forward command with $v_y = \omega_z = 0$.
Leave `lin_vel_y` and `ang_vel_z` as they are: they are overridden anyway.

Things to think about:
- Which speeds are realistic on stairs?
- Should robots sometimes be commanded to stand still on a staircase?

In [ ]:
def make_commands(play: bool) -> dict[str, CommandTermCfg]:
    return {
        "joystick": mdp.UniformVelocityCommandCfg(
            # One command per episode (longer than the 20 s episode): the curriculum compares the
            # walked distance with the command at the end of the episode, so a command that
            # changes during the episode would make this check meaningless
            resampling_time_range=(1e3, 1e3),
            # No standing robots: a robot with a zero command never moves, so the curriculum
            # can neither promote nor demote it, and it never reaches the steps anyway
            # (spawn on the flat central platform)
            rel_standing_envs=0.0,
            rel_forward_envs=1.0, # IMPORTANT! This notebook does FORWARD locomotion only!
            ranges=mdp.UniformVelocityCommandCfg.Ranges(
                # 0.5 to 1.0 m/s: tracked well by the flat policy and moderate for 20 cm steps.
                # The lower bound lets a robot reach the terrain edge (5.7 m from the centre)
                # within 20 s even at 80 % tracking (0.4 m/s x 20 s = 8 m), so it can be promoted.
                # Forward robots get |vx| clamped to at least 0.3 m/s by mjlab, so keep it positive.
                lin_vel_x=(0.5, 1.0), # m/s
                lin_vel_y=(-1.0, 1.0), # Fix for mjlab viewer (min and max have to be different)...
                ang_vel_z=(-1.0, 1.0), # ...rel_forward_envs will override this. Do not change.
            ),
            entity_name="robot",
            debug_vis=play, # Do not render command arrows during training
        )
    }

### 3.1b Observations

Two changes compared to Task 1.

**Terrain scan.**
The sensor `"terrain_scan"` casts a grid of rays (1.6 m $\times$ 1.0 m around the base, every 0.2 m) down onto the terrain.
`mdp.height_scan` returns, for each ray, the height of the base above the terrain at that point.
This enables the policy to see steps ahead.
You can see the rays in the viewer.

**Observation noise.**
Each term can now get uniform noise with `noise=UniformNoiseCfg(n_min=..., n_max=...)`, added to the observation at every step.
Noise is only applied in groups with `enable_corruption=True`.
Choose noise levels that roughly match the real sensors: joint encoders are precise, IMU angular velocities less so, and an elevation map can be very noisy (several centimetres).

Things to think about:
- Can the policy learn without the terrain scan (i.e., blind locomotion)? How would that policy differ?
- Should all observations, including the command, be noisy for robustness?
- Should the critic receive noisy observations?
- How would a terrain scan be obtained on a real robot?

In [ ]:
def noise(level: float) -> UniformNoiseCfg:
    """Uniform noise in [-level, level], added at every step."""
    return UniformNoiseCfg(n_min=-level, n_max=level)


def policy_terms(noisy: bool) -> dict[str, ObservationTermCfg]:
    """Terms the robot can measure, with sensor noise if `noisy`.

    Built by a function so the actor and the critic get separate term objects:
    mjlab removes the noise from the terms of a group without corruption, which would
    also remove it from the actor if both groups shared the same objects.
    """
    n = noise if noisy else (lambda level: None)
    return {
        # Command: set by the operator, known exactly, so no noise
        "command": ObservationTermCfg(
            func=mdp.generated_command,
            params={"command_name": "joystick"},
        ),
        # Base state from the IMU and the state estimator (same terms as the flat MDP)
        "base_lin_vel": ObservationTermCfg(func=mdp.base_lin_vel_b, noise=n(0.1)),  # m/s, estimator drift
        "base_ang_vel": ObservationTermCfg(func=mdp.base_ang_vel_b, noise=n(0.2)),  # rad/s, gyroscope
        "projected_gravity": ObservationTermCfg(func=mdp.projected_gravity_b, noise=n(0.05)),  # IMU attitude
        # Joint encoders: positions are precise, velocities are differentiated and noisier
        "joint_pos": ObservationTermCfg(func=mdp.joint_positions, noise=n(0.01)),  # rad
        "joint_vel": ObservationTermCfg(func=mdp.joint_velocities, noise=n(1.5)),  # rad/s
        # Previous action: computed by the policy itself, known exactly
        "last_action": ObservationTermCfg(func=mdp.last_action),
        # Terrain scan (54 rays): base height above the terrain around the robot.
        # Noise of a few centimetres, like an elevation map built from depth cameras or lidar.
        # Clip only bounds the 5 m value of a missed ray; real heights stay within it.
        "height_scan": ObservationTermCfg(
            func=mdp.height_scan,
            params={"sensor_name": scene.TERRAIN_SCAN_SENSOR},
            noise=n(0.05),  # m
            clip=(-1.0, 1.5),
        ),
    }


def make_observations(play: bool) -> dict[str, ObservationGroupCfg]:
    actor = policy_terms(noisy=True)

    critic = {
        # Same terms as the actor, without noise: the critic is only used in simulation,
        # so it can use the true state for a better value estimate
        **policy_terms(noisy=False),
        # Privileged foot contact information, as in the flat MDP
        "foot_contact": ObservationTermCfg(
            func=mdp.foot_contact,
            params={"sensor_name": scene.FEET_CONTACT_SENSOR},
        ),
        "foot_air_time": ObservationTermCfg(
            func=mdp.foot_air_time,
            params={"sensor_name": scene.FEET_CONTACT_SENSOR},
        ),
    }

    return {
        # Noise on the actor only when the flag is on (run 2)
        "actor": ObservationGroupCfg(terms=actor, enable_corruption=USE_NOISE),
        "critic": ObservationGroupCfg(terms=critic, enable_corruption=False),
    }

### 3.1c Actions

Same action space as in Task 1.

Things to think about:
- Does the action scale that worked on flat ground allow the legs to lift high enough for a step?

In [ ]:
def make_actions(play: bool) -> dict[str, ActionTermCfg]:
    return {
        "joint_targets": mdp.JointPositionActionCfg(
            entity_name="robot",
            actuator_names=(".*"), # all 12 joints
            use_default_offset=True, # action 0 = default standing pose
            # Same scale as the flat MDP (validated there; Rudin et al. also climb stairs with 0.25).
            # The policy output is a Gaussian, not bounded to [-1, 1], so larger swings stay possible.
            # If the feet catch the step edges in the viewer, a larger scale is the first fix to try.
            scale=0.25, # rad of joint motion per unit of action
        ),
    }

### 3.1d Rewards

Start from your Task 1 rewards, then tune each term against the new task.
Some penalties that made sense on flat ground might work against going on stairs.
For instance, does it make sense to penalize vertical base velocity when climbing up a stair?

Things to think about:
- Which penalties should be reduced or removed on stairs?
- Watch `Episode_Reward/<term>` and the terrain level (3.3), which dominates?

In [ ]:
def make_rewards(play: bool) -> dict[str, RewardTermCfg]:
    return {
        # Task (same as flat)
        "track_linear_velocity": RewardTermCfg(
            func=mdp.track_linear_velocity,
            weight=1.0,
            params={"command_name": "joystick", "std": math.sqrt(0.25)},
        ),
        # The yaw rate command is always 0 here: this term now keeps the robot walking straight
        "track_angular_velocity": RewardTermCfg(
            func=mdp.track_angular_velocity,
            weight=0.5,
            params={"command_name": "joystick", "std": math.sqrt(0.25)},
        ),
        # Base stability, relaxed for stairs:
        # vertical velocity is required to climb or descend (about 0.3 m/s at 0.5 m/s on
        # 20 cm x 30 cm steps), so -2.0 (flat) -> -0.5
        "lin_vel_z": RewardTermCfg(func=mdp.body_linear_velocity_penalty, weight=-0.5),
        "ang_vel_xy": RewardTermCfg(func=mdp.body_angular_velocity_penalty, weight=-0.05),
        # the base has to pitch to follow the slope of the stairs, so -1.0 (flat) -> -0.2,
        # kept small to still discourage large tilts when stepping down
        "flat_orientation": RewardTermCfg(func=mdp.flat_orientation_l2, weight=-0.2),
        # Smoothness and efficiency (same as flat)
        "joint_torques": RewardTermCfg(func=mdp.joint_torques_l2, weight=-1.0e-5),
        "joint_acc": RewardTermCfg(func=mdp.joint_acc_l2, weight=-2.5e-7),
        "action_rate": RewardTermCfg(func=mdp.action_rate_l2, weight=-0.01),
        # Thigh, hip or calf hitting the ground or a step edge: on stairs this also pushes
        # the policy to lift the feet clear of the edges
        "leg_contacts": RewardTermCfg(
            func=mdp.contacts_penalty,
            weight=-1.0,
            params={"sensor_name": scene.LEG_CONTACT_SENSOR},
        ),
        # Shaping: reward stepping instead of shuffling (same as flat; the command is
        # always above the 0.5 m/s threshold here, so it is always active)
        "feet_air_time": RewardTermCfg(
            func=mdp.feet_air_time,
            weight=0.25,
            params={
                "sensor_name": scene.FEET_CONTACT_SENSOR,
                "command_name": "joystick",
                "threshold_min": 0.05,
                "threshold_max": 0.5,
            },
        ),
    }

### 3.1e Terminations

Same principle as in Task 1: a time-out, plus at least one failure termination that detects a fall.
We just add one more termination, when the robot reaches the end of their terrain (`terrain_edge_reached`).

In [ ]:
def make_terminations(play: bool) -> dict[str, TerminationTermCfg]:
    return {
        # End of the 20 s episode: not a failure, the robot could keep walking
        "time_out": TerminationTermCfg(
            func=mdp.time_out,
            time_out=True,
        ),
        # End of the 12 m tile: this is a success, not a failure. As a time-out, PPO keeps
        # bootstrapping the value of the last state instead of treating it as a dead end,
        # otherwise the robot would learn to avoid reaching the edge
        "terrain_edge_reached": TerminationTermCfg(
            func=mdp.terrain_edge_reached,
            time_out=True,
        ),
        # Fall: the base (trunk) touches the ground or a step
        "base_contact": TerminationTermCfg(
            func=mdp.illegal_contact,
            params={"sensor_name": scene.BASE_CONTACT_SENSOR},
            time_out=False,
        ),
        # Fall: base tilted by more than ~57 degrees. Kept as on flat ground: the base
        # pitch needed on 20 cm x 30 cm stairs stays well below this limit
        "bad_orientation": TerminationTermCfg(
            func=mdp.bad_orientation,
            params={"limit_angle": 1.0},
            time_out=False,
        ),
    }

### 3.1f Events

**Initial state.**
Each robot starts at the centre of a staircase (on its top or bottom platform, 1.5 m wide).
Choose the initial pose range so robots do not spawn partly inside a step.

**Domain randomization.**
The simulation model is never exactly the real robot.
Modern deep reinforcement learning methods rely on domain randomization, where physical properties of the robot and environment are randomized.
The policy has to learn to work for the whole range of robots with different robots, and, hopefully, for the real one.

Here you randomize the **centre of mass of the base**, with `mdp.body_com_offset`:

- `operation`: how the sampled value is applied: `"add"` to the nominal value, `"scale"` it, or replace it (`"abs"`)
- `ranges`: (min, max) per axis

Things to think about:
- When should the randomization be sampled?
- How large can the ranges be before the task becomes impossible?
- Which other quantities would be worth randomizing? (no need to implement them)

In [ ]:
def make_events(play: bool) -> dict[str, EventTermCfg]:
    events = {
        "reset_base": EventTermCfg(
            func=mdp.reset_root_pose_uniform,
            mode="reset", # at the start of every episode
            params={
                "pose_range": {
                    # The central platform is 1.5 m wide (+-0.75 m). With any heading the
                    # feet stay within about 0.35 m of the base, so +-0.3 m keeps all feet
                    # on the platform (0.65 m < 0.75 m)
                    "x": (-0.3, 0.3), # m
                    "y": (-0.3, 0.3), # m
                    "z": (0.0, 0.05), # m, slightly above the default height to avoid penetration
                    "roll": (-0.1, 0.1), # rad
                    "pitch": (-0.1, 0.1), # rad
                    # Any heading: the pyramids are symmetric, so the robot meets the steps
                    # from every direction, including diagonally across the corners
                    "yaw": (-math.pi, math.pi), # rad
                }
            },
        ),
        # Reset joints to the nominal standing pose, otherwise a robot keeps the
        # (possibly fallen) joint configuration of its previous episode
        "reset_joints": EventTermCfg(
            func=mdp.reset_joints_by_offset,
            mode="reset",
            params={
                "position_range": (0.0, 0.0), # rad
                "velocity_range": (0.0, 0.0), # rad/s
            },
        ),
    }
    if USE_DR:
        # Domain randomization: offset of the base centre of mass from its nominal position.
        # "startup": sampled once per robot, so the 4096 robots are 4096 different bodies,
        # like a real robot whose payload or battery position is fixed during a run.
        # "add": offset added to the nominal value (mjlab adds it to the default, so it
        # never accumulates). Ranges: a few cm, like a shifted payload on a 15 kg robot.
        # Larger shifts in x load the front or rear legs, whose thighs are already near
        # torque saturation on flat ground (22.7 of 23.5 N m).
        events["base_com"] = EventTermCfg(
            mode="startup",
            func=mdp.body_com_offset,
            params={
                "asset_cfg": SceneEntityCfg("robot", body_names=("base_link",)),
                "operation": "add",
                "ranges": {
                    0: (-0.05, 0.05), # m, forward / backward
                    1: (-0.03, 0.03), # m, left / right
                    2: (-0.03, 0.03), # m, up / down
                },
            },
        )
    return events

### 3.1g Terrain curriculum

The stairs terrain is a grid of staircases: 10 rows of increasing difficulty (step height from 0 cm to 20 cm) and 2 columns (pyramids to walk down, and inverted pyramids to climb up).
Each robot is assigned a row, its **terrain level**.

Without a curriculum, most robots would face stairs they cannot climb yet: they fall immediately, collect almost no reward, and learn nothing.
With `terrain_levels_vel`, the level adapts to each robot's performance at the end of every episode:

- **promoted** to a harder row if it walked at least 80 % of the distance its command asked for OR it reached the end of the terrain
- **demoted** to an easier row if it walked less than half of that distance.

Follow `Curriculum/terrain_levels/...` in TensorBoard: a rising mean level means the robots master higher steps.
Training has succeeded when most robots stay on the hardest levels.

Things to think about:
- Why does the curriculum use the walked distance rather than the reward?
- What happens if robots are commanded to stand still?
- What other types of curriculums can you think about? (no need to implement them)

In [ ]:
# Provided curriculum, unchanged. How it behaves in mjlab 1.6 (useful to read the curves):
# - promotion: walked more than 6 m from the spawn point, or the terrain edge was reached;
#   demotion: walked less than half of |command| x elapsed time
# - initial levels are uniform over the 10 rows, so the mean level starts around 4.5
# - a robot promoted beyond the last row is sent back to a random row, so even a policy
#   that succeeds everywhere settles around a mean level of 6, not 9
def make_curriculum(play: bool) -> dict[str, CurriculumTermCfg]:
    return {
        "terrain_levels": CurriculumTermCfg(
            func=mdp.terrain_levels_vel,
            params={"command_name": "joystick", "edge_term_name": "terrain_edge_reached"},
        ),
    }

### 3.1h Assembly

As in Task 1, with `terrain="stairs"` and the new `curriculum` part.

In [ ]:
def make_env_cfg(play: bool = False, seed: int = 42) -> ManagerBasedRlEnvCfg:
    return ManagerBasedRlEnvCfg(
        scene=scene.make_scene_cfg(
            num_envs=1 if play else 4096,
            terrain="stairs", # We are using the stairs environment this time!
            robot_entity_name="robot",
        ),
        sim=scene.make_sim_cfg(timestep_s=0.005),
        actions=make_actions(play),
        commands=make_commands(play),
        curriculum=make_curriculum(play),
        events=make_events(play),
        observations=make_observations(play),
        rewards=make_rewards(play),
        terminations=make_terminations(play),
        decimation=4,
        episode_length_s=20.0,
        seed=seed,
    )

### 3.1i Visualize your environment

Check that robots spawn correctly on the stairs (make sure to pause and reset), that the terrain scan rays look right, and that a fallen robot is reset.
Stop the viewer with the **interrupt** button (■) before running the next cell.

In [ ]:
env_cfg = make_env_cfg(play=True)
play.interactive(env_cfg, policy="random") # "random" or "zero"

## 3.2 Building the training configuration

Same networks and PPO settings as in Task 1.
While better performance can be obtained by tuning these, we recommend keeping the parameters we tuned for you.

In [ ]:
def make_actor_cfg() -> RslRlModelCfg:
    return RslRlModelCfg(
        hidden_dims=(128, 128, 128),
        activation="elu",
        obs_normalization=True,
        distribution_cfg={
            "class_name": "GaussianDistribution",
            "init_std": 1.0,
            "std_type": "scalar",
        },
    )

def make_critic_cfg() -> RslRlModelCfg:
    return RslRlModelCfg(
        hidden_dims=(128, 128, 128),
        activation="elu",
        obs_normalization=True,
    )

def make_ppo_cfg() -> RslRlPpoAlgorithmCfg:
    return RslRlPpoAlgorithmCfg(
        value_loss_coef=1.0,
        use_clipped_value_loss=True,
        clip_param=0.2,
        entropy_coef=0.01,
        num_learning_epochs=5,
        num_mini_batches=4,
        learning_rate=1.0e-3,
        schedule="adaptive",
        gamma=0.995,
        lam=0.97,
        desired_kl=0.01,
        max_grad_norm=1.0,
    )

def make_rl_cfg(seed: int = 42) -> RslRlOnPolicyRunnerCfg:
    return RslRlOnPolicyRunnerCfg(
        seed=seed,
        num_steps_per_env=24,
        max_iterations=500,
        save_interval=500,
        actor=make_actor_cfg(),
        critic=make_critic_cfg(),
        algorithm=make_ppo_cfg(),
    )

## 3.3 Training

Start TensorBoard and open the printed link.
Besides the reward and episode length curves, follow the terrain level (`Curriculum/terrain_levels/...`) to see how far up the curriculum the robots get.
The episode length is less informative than on flat ground: a robot that reaches the edge of its tile ends its episode early, which is a success.

In [ ]:
train.tensorboard()

The training cell skips every seed of `VARIANT` that already has a complete run, so running it again after a kernel restart does not retrain by accident.

In [ ]:
def run_seed(r) -> int:
    """Seed of a run, read from the config saved with it."""
    return yaml.unsafe_load((r.path / "params" / "agent.yaml").read_text())["seed"]


def runs_per_seed(variant: str) -> dict:
    """Latest complete run of `variant` for each seed in SEEDS (seed -> Run)."""
    last = f"model_{make_rl_cfg().max_iterations - 1}.pt"
    found = {}
    for r in registry.runs(experiment="stairs", variant=variant):  # sorted by creation time
        if (r.path / last).exists() and run_seed(r) in SEEDS:
            found[run_seed(r)] = r
    return found

In [ ]:
for seed in SEEDS:
    if seed in runs_per_seed(VARIANT):
        print(f"{VARIANT}, seed {seed}: already trained, skipped")
        continue
    train.train(
        env_cfg=make_env_cfg(seed=seed),
        rl_cfg=make_rl_cfg(seed=seed),
        experiment="stairs",
        variant=VARIANT,
    )

### Loading the latest run

Works after a kernel restart without retraining.

In [ ]:
stairs_runs = runs_per_seed(VARIANT)  # seed -> run
run = stairs_runs[42]
print(f"Found checkpoint at: {run.checkpoint}")

## 3.4 Testing your policy

Watch your policy in the viewer and check that it crosses stairs **reliably**: up and down, over many episodes rather than once.
If it fails, note *how*: tripping on step edges, refusing to move, slipping, falling when stepping down.
Each failure points to a different fix in the observations, rewards or terminations.

The demo uses one robot, no 20 s time-out, a single staircase type and the same step height on every row, so up and down can be shown separately at a chosen height.
The robot is still reset when it reaches the edge of the tile.

In [ ]:
DEMO_DIRECTION = "up"  # "up": climb out of the inverted pyramid, "down": walk down the pyramid
DEMO_STEP_HEIGHT = 0.20  # m, used on every row

demo_cfg = make_env_cfg(play=True)
demo_cfg.episode_length_s = 1e6  # demo only: no 20 s time-out reset in the viewer
generator = demo_cfg.scene.terrain.terrain_generator
keep = "stairs_up" if DEMO_DIRECTION == "up" else "stairs_down"
generator.sub_terrains = {keep: generator.sub_terrains[keep]}
generator.sub_terrains[keep].proportion = 1.0
generator.sub_terrains[keep].step_height_range = (DEMO_STEP_HEIGHT, DEMO_STEP_HEIGHT)
generator.num_cols = 1

play.interactive(
    env_cfg=demo_cfg,
    rl_cfg=make_rl_cfg(),
    policy=run.checkpoint,
    num_envs=1,
)